# iter_4 Macro Autoresearch Analysis

Validation CAGR is the selection objective subject to no ruin. Locked OOS and DBMF excess are diagnostics only.


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
results = pd.read_csv(Path('results.tsv'), sep='\t')
numeric_cols = ['train_cagr', 'validation_cagr', 'oos_cagr', 'benchmark_oos_cagr', 'excess_oos_cagr_vs_dbmf']
for col in numeric_cols:
    results[col] = pd.to_numeric(results[col], errors="coerce")
results['trial'] = range(1, len(results) + 1)
results['ruined_flag'] = results['ruined'].astype(str).str.lower().isin(['true', '1'])
eligible = results[(~results['ruined_flag']) & (results['validation_cagr'] > 0)].copy()
best = eligible.loc[eligible['validation_cagr'].idxmax()]
keeps = results[results['status'] == 'keep'].copy()
summary = pd.DataFrame([{'rows': len(results), 'keeps': len(keeps), 'ruined': int(results['ruined_flag'].sum()), 'best_idea_id': best['idea_id'], 'best_validation_cagr': best['validation_cagr'], 'best_train_cagr': best['train_cagr'], 'best_oos_cagr': best['oos_cagr'], 'best_benchmark_oos_cagr': best['benchmark_oos_cagr'], 'best_excess_oos_cagr_vs_dbmf': best['excess_oos_cagr_vs_dbmf']}])
summary


In [ ]:
recent20 = results.tail(20).copy()
recent_summary = pd.DataFrame([{
    'added_trials': len(recent20),
    'added_keeps': int((recent20['status'] == 'keep').sum()),
    'best_added_idea_id': recent20.loc[recent20['validation_cagr'].idxmax(), 'idea_id'],
    'best_added_validation_cagr': recent20['validation_cagr'].max(),
    'best_added_oos_cagr': recent20.loc[recent20['validation_cagr'].idxmax(), 'oos_cagr'],
}])
display(recent_summary)
recent20[['trial', 'idea_id', 'status', 'train_cagr', 'validation_cagr', 'oos_cagr', 'benchmark_oos_cagr', 'excess_oos_cagr_vs_dbmf', 'ruined']]

In [ ]:
display_cols = ['trial', 'idea_id', 'status', 'train_cagr', 'validation_cagr', 'oos_cagr', 'benchmark_oos_cagr', 'excess_oos_cagr_vs_dbmf', 'ruined', 'description']
results.sort_values('validation_cagr', ascending=False)[display_cols].head(12)


In [ ]:
plot = results[['trial', 'idea_id', 'status', 'validation_cagr', 'oos_cagr', 'benchmark_oos_cagr']].copy()
plot['running_best_validation_cagr'] = plot['validation_cagr'].where(~results['ruined_flag']).cummax()
fig, ax = plt.subplots(figsize=(12, 7))
colors = plot['status'].map({'keep': '#2ca02c', 'discard': '#7f7f7f', 'ruined': '#d62728', 'invalid_rationale': '#ff7f0e'}).fillna('#7f7f7f')
ax.scatter(plot['trial'], plot['validation_cagr'], c=colors, s=54, label='trial validation CAGR')
ax.plot(plot['trial'], plot['running_best_validation_cagr'], color='#1f77b4', linewidth=2.5, label='running best validation CAGR')
ax.axhline(float(best['benchmark_oos_cagr']), color='#9467bd', linestyle='--', linewidth=1.5, label='DBMF locked-OOS CAGR diagnostic')
ax.scatter([int(best['trial'])], [float(best['validation_cagr'])], color='gold', edgecolor='black', s=180, zorder=5, label='best validation-selected run')
ax.annotate(str(best['idea_id']), xy=(int(best['trial']), float(best['validation_cagr'])), xytext=(int(best['trial']) - 12, float(best['validation_cagr']) + 0.035), arrowprops={'arrowstyle': '->', 'color': 'black'}, fontsize=9)
ax.set_title('iter_4 validation-CAGR progress after 20 added trials; OOS/DBMF shown only as diagnostics')
ax.set_xlabel('ledger trial')
ax.set_ylabel('CAGR')
ax.grid(True, alpha=0.25)
ax.legend(loc='best')
fig.tight_layout()
fig.savefig('progress.png', dpi=180)
plt.show()


## Best run interpretation

The selected run is the highest non-ruined validation-CAGR strategy in the ledger. OOS CAGR and DBMF excess are locked diagnostics, not the selection criterion.


In [ ]:
best[['idea_id', 'title', 'mechanism', 'expected_assets', 'feature_inputs', 'human_notes', 'train_cagr', 'validation_cagr', 'oos_cagr', 'benchmark_oos_cagr', 'excess_oos_cagr_vs_dbmf', 'ruined', 'status']].to_frame('best_run')


## Data quality, robustness, and anti-overfit audit

This section is intentionally user-facing. It gives a quick, auditable view of whether a candidate result is likely a fragile quant-research artifact:

- **Data quality:** first/last valid price dates, observation counts, missing counts, and longest missing streaks. Pre-inception gaps are visible instead of silently treated as zero returns.
- **Robustness:** parameter-neighborhood summaries should be read by median/p10/p90 and trial count, not by the single best validation row.
- **Anti-overfit:** trial count, family count, and best-minus-median validation performance show selection pressure. Locked OOS/DBMF diagnostics are for post-selection audit only.


In [ ]:
from pathlib import Path

import pandas as pd

from datasets.quality import price_coverage_manifest
from research.overfit import selection_bias_summary
from research.robustness import summarize_robust_regions

artifact_paths = {
    "robustness": Path("robustness.tsv"),
    "overfit": Path("overfit_report.tsv"),
    "results": Path("results.tsv"),
}

try:
    from prepare import DEFAULT_UNIVERSE, load_prices
    prices = load_prices(DEFAULT_UNIVERSE)
    display(price_coverage_manifest(prices).sort_values(["longest_missing_streak", "missing_count"], ascending=False))
except Exception as exc:
    print(f"Price coverage manifest unavailable: {exc}")

if artifact_paths["robustness"].exists():
    robustness = pd.read_csv(artifact_paths["robustness"], sep="\t")
    group_columns = [column for column in ["universe_name", "lookback_days", "top_n", "score_weighted", "risk_cap"] if column in robustness.columns]
    if group_columns:
        display(summarize_robust_regions(robustness, group_columns).head(20))
else:
    print("robustness.tsv not found yet. Generate it with research.robustness.run_robustness_grid and write_robustness_tsv.")

if artifact_paths["results"].exists():
    results = pd.read_csv(artifact_paths["results"], sep="\t")
    display(pd.DataFrame([selection_bias_summary(results)]))
else:
    print("results.tsv not found yet. Anti-overfit summary will appear after experiment rows exist.")


## Public data artifact and feature-usage audit

These cells make the text/macro-data path auditable: the manifest shows which public artifacts were created, and the ledger display checks whether non-price feature claims produced parseable usage lines.

In [ ]:
manifest_path = Path.home() / '.cache' / 'macroresearch' / 'iter_4' / 'artifact_manifest.tsv'
if manifest_path.exists():
    manifest = pd.read_csv(manifest_path, sep='\t')
    display(manifest[['dataset', 'row_count', 'artifact_path', 'sha256', 'point_in_time_safe']])
else:
    print(f'needs_data: missing artifact manifest at {manifest_path}')


In [ ]:
usage_rows = results[results['description'].fillna('').str.contains('feature_usage_columns:', regex=False)].copy()
usage_display_cols = [column for column in ['trial', 'idea_id', 'status', 'feature_inputs', 'validation_cagr', 'description'] if column in usage_rows.columns]
if usage_rows.empty:
    print('No public feature usage rows found.')
else:
    display(usage_rows[usage_display_cols].tail(10))
